# Experimental Models - Buy Price Prediction

In [1]:

import os

from dotenv import load_dotenv

from mlModels.kmeans.runCluster import runCluster
from mlModels.regression.data.data import getData, getRegressionData

CV = True
CV_FOLDS = 5

os.chdir("/home/florian/Desktop/immopreis-regression")

load_dotenv("database/.env")

df_features = getData(filter_type="finance_type", filter_val="rent", table="buy_features")
cluster_dict = runCluster()

/home/florian/Desktop/immopreis-regression/mlModels/regression/data/data.py:16: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df_features = pd.read_sql("SELECT * FROM rent_features", conn)
/home/florian/Desktop/immopreis-regression/mlModels/regression/data/data.py:17: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df_listings = pd.read_sql("SELECT * FROM listings", conn)
/home/florian/Desktop/immopreis-regression/mlModels/kmeans/data/data.py:7: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df = pd.read_sql("SELECT * FROM lis


-------Starting K-Means Clustering locations-------


# XGBoost

In [3]:
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import numpy as np
import pandas as pd
from xgboost import XGBRegressor as xgbr

for k, df_cluster in cluster_dict.items():
    df = df_features.merge(df_cluster, on="id", how="inner")
    df_house_X, df_house_y, df_apt_X, df_apt_y = getRegressionData(df)

    drop_cols = ["is_dgw", "is_egw", "is_gc", "is_gw", "is_ms", "is_phw", "is_apt", "is_wg"]

    df_apt_X.drop(columns=drop_cols,  inplace=True, errors="ignore")
    df_apt_y.drop(columns=drop_cols, inplace=True, errors="ignore")

    X_train, X_test, y_train, y_test = train_test_split(df_apt_X, df_apt_y, test_size=0.2, random_state=42)

    drop_cols = ["finance_type", "loc_14_7", "loc_14_8", "is_ab", "is_rh", "is_dhh",
                 "is_sbc", "is_villa", "log_wintergarden_size", "is_gh", "is_bh",
                 "loc_14_6", "wintergarden_ratio", "estate_ratio", "fgee", "is_ceiling", "has_kitchen", "is_bio", "is_electro", "loc_14_10"]

    X_train.drop(columns=drop_cols, inplace=True, errors="ignore")
    X_test.drop(columns=drop_cols, inplace=True, errors="ignore")

    model = xgbr(
        n_estimators=300,
        max_depth=5,
        learning_rate=0.05,
        subsample=0.8,
        colsample_bytree=0.8,
        random_state=42,
        n_jobs=-1
    )

    if CV:
        scores = cross_val_score(model, X_train, y_train, cv=CV_FOLDS, scoring="r2", verbose=True)
        print(f"\n-------Cross Validation Scores for K={k}-------")
        print(f"CV R2 Scores: {scores}")
        print(f"Mean R2 Score: {scores.mean():.4f}")
        print(f"Standard Deviation: {scores.std():.4f}")
        print()

    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)

    mae = mean_absolute_error(y_test, y_pred)
    rmse = np.sqrt(mean_squared_error(y_test, y_pred))
    r2 = r2_score(y_test, y_pred)

    print(f"-------XGBoost Regression Results for K={k}-------")
    print(f"MAE:  {mae:.4f}")
    print(f"RMSE: {rmse:.4f}")
    print(f"R2:   {r2:.4f}")
    importance_df = pd.DataFrame({
        "feature" : X_train.columns,
        "importance" : model.feature_importances_
    })
    importance_df = importance_df.sort_values("importance", ascending=False)
    print(importance_df.tail(20))
    print()

[Parallel(n_jobs=1)]: Done   5 out of   5 | elapsed:    0.9s finished



-------Cross Validation Scores for K=3-------
CV R2 Scores: [0.59474363 0.6445588  0.62682486 0.72315507 0.62147729]
Mean R2 Score: 0.6422
Standard Deviation: 0.0435

-------XGBoost Regression Results for K=3-------
MAE:  0.1412
RMSE: 0.1746
R2:   0.7160
            feature  importance
4        has_cellar    0.012610
5       has_parking    0.012015
10       has_loggia    0.011275
13       is_pellets    0.011121
7       has_balcony    0.010539
22     garden_ratio    0.008862
18       is_central    0.008615
33  log_loggia_size    0.008360
12           is_oil    0.007452
0             rooms    0.006930
6        has_closet    0.006131
31  log_garden_size    0.005521
15    is_geothermal    0.005041
24         is_urban    0.005038
16   is_air_heating    0.004998
14  is_photovoltaik    0.004933
20      is_infrared    0.002602
26           is_efh    0.000000
27            is_lh    0.000000
25           is_mfh    0.000000



[Parallel(n_jobs=1)]: Done   5 out of   5 | elapsed:    1.1s finished



-------Cross Validation Scores for K=4-------
CV R2 Scores: [0.57289177 0.66014788 0.68442031 0.71655122 0.58963785]
Mean R2 Score: 0.6447
Standard Deviation: 0.0551

-------XGBoost Regression Results for K=4-------
MAE:  0.1349
RMSE: 0.1650
R2:   0.7461
            feature  importance
31  log_garden_size    0.014941
40          loc_4_2    0.014062
33  log_loggia_size    0.013525
13       is_pellets    0.013429
18       is_central    0.012317
10       has_loggia    0.012205
22     garden_ratio    0.011405
15    is_geothermal    0.010396
0             rooms    0.010251
6        has_closet    0.009264
12           is_oil    0.008970
14  is_photovoltaik    0.007002
16   is_air_heating    0.006965
38          loc_4_0    0.006843
24         is_urban    0.006546
39          loc_4_1    0.006324
20      is_infrared    0.002151
25           is_mfh    0.000000
27            is_lh    0.000000
26           is_efh    0.000000


-------Cross Validation Scores for K=14-------
CV R2 Scores: [0.572578

[Parallel(n_jobs=1)]: Done   5 out of   5 | elapsed:    1.5s finished
